In [1]:
from dotenv import load_dotenv
import os
import time
import json
import requests
import pandas as pd

# Ruta absoluta o relativa al .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de Gemini desde .env
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if GEMINI_API_KEY:
    print("Clave de Gemini cargada correctamente.")
else:
    raise ValueError("No se encontró la clave de Gemini. Verifica la ruta del .env.")

Clave de Gemini cargada correctamente.


In [2]:
def call_gemini_api(prompt, text):
    """
    Sends a text and a prompt to Gemini 1.5 Flash and returns the generated summary.
    """
    url = f"https://generativelanguage.googleapis.com/v1/models/gemini-2.5-flash:generateContent?key={GEMINI_API_KEY}"

    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "contents": [
            {
                "parts": [
                    {"text": f"{prompt}\n\nText:\n{text}"}
                ]
            }
        ],
        "generationConfig": {
            "temperature": 0.7,
            "maxOutputTokens": 8192
        }
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        print(f"Status code: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            print("Response keys:", data.keys())
            
            # Debug: Ver la estructura completa
            if "candidates" in data:
                print("Candidates keys:", data["candidates"][0].keys())
                if "content" in data["candidates"][0]:
                    print("Content keys:", data["candidates"][0]["content"].keys())
            
            # Intentar extraer el texto con mejor manejo de errores
            try:
                output = data["candidates"][0]["content"]["parts"][0]["text"]
                return output.strip(), elapsed
            except KeyError as ke:
                print(f"KeyError: {ke}")
                print("Estructura completa de la respuesta:")
                print(json.dumps(data, indent=2)[:500])
                return None, elapsed
        else:
            print(f"HTTP error {response.status_code}: {response.text[:200]}")
            return None, elapsed

    except Exception as e:
        print("Request error:", e)
        import traceback
        traceback.print_exc()
        return None, None

In [3]:
prompt = """Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words.

    Abstract of a biomedical study text:Using the following abstract of a biomedical study as input, generate a Plain Language Summary (PLS) understandable by any patient, regardless of their health literacy. Ensure that the generated text adheres to the following instructions which should be followed step-by-step:
    a. Specific Structure: the generated PLS should be presented in a logical order, using the following order:
        1. Plain Title
        2. Rationale
        3. Trial Design
        4. Results
    b. Sections should be authored following these parameters:
        1. Plain Title: Simplified title understandable to a layperson that summarizes the research that was done.
        2. Rationale: Include: background or study rationale providing a general description of the condition, what it may cause or why it is a burden for the patients; the reason and main hypothesis for the study; and why the study is needed, and why the study medication has the potential to treat the condition.
        3. Trial Design: Answer ‘How is this study designed?’ Include the description of the design, description of study and patient population (age, health condition, genre), and the expected amount of time a person will be in the study.
        4. Results: answer ‘What were the main results of the study’, include what are the benefits for the patients, how the study was relevant for the area of study, and what are the conclusions from the investigator.
    c. Consistency and Replicability: the generated PLS should be consistent regardless of the order of sentences or the specific phrasing used in the input protocol text.
    d. Compliance with Plain Language Guidelines: The generated PLS must follow all of these plain language guidelines:
        1. Have readability grade level of 6 or below.
        2. Do not have jargon. All technical or medical words or terms should be defined or broken down into simple and logical explanations.
        3. Active voice, not passive
        4. Mostly one or two syllable words
        5. Sentences of 15 words or less
        6. Short paragraphs of 3-5 sentences
        7. Simple numbers (eg, ratios, no percentages)
    e. Do not invent Content: The AI model should not invent information. If the AI model includes data other than the one given in the input abstract, the AI model should guarantee such data is verified and real.
    f. Aim for an approximate PLS length of 500-900 words."""
text = "The administration of statins has been shown to reduce LDL cholesterol and cardiovascular risk."
resumen, tiempo = call_gemini_api(prompt, text)
if resumen:
    print(resumen)
    print(f"Time: {tiempo:.2f} s")

Status code: 200
Response keys: dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])
Candidates keys: dict_keys(['content', 'finishReason', 'index'])
Content keys: dict_keys(['parts', 'role'])
The abstract you provided is extremely limited: "The administration of statins has been shown to reduce LDL cholesterol and cardiovascular risk."

This single sentence states a general, known medical fact about statins. It does not describe a specific biomedical study, clinical trial, or research project.

Therefore, I cannot generate a complete Plain Language Summary (PLS) that adheres to all your instructions, specifically:
*   **Rationale:** I cannot include the specific reason and main hypothesis for *the study*, why *the study* is needed, or why *the study medication* has the potential to treat a condition *within the context of a specific study*, because no specific study is described. I can only speak generally about statins.
*   **Trial Design:** I cannot describe the 

In [4]:
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [5]:
# Preparar columna para resúmenes generados
df["gen_summary"] = ""

# Ruta donde se guardará el CSV con los resultados
ruta_csv = "./results_gemini_v2.csv"

# Procesar cada artículo
for i, fila in df.iterrows():
    print(f"\nProcessing {fila['name']} ({i+1}/{len(df)})...\n")
    resumen, tiempo = call_gemini_api(prompt, fila["article"])
    
    if resumen:
        df.loc[i, "gen_summary"] = resumen
        print(f"Response time: {tiempo:.2f} s\n")
    else:
        print("No response received.\n")

# Guardar resultados
df.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Results saved to: {os.path.abspath(ruta_csv)}")
display(df.head())


Processing 10.1002-14651858.CD009781.pub2 (1/380)...

Status code: 200
Response keys: dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])
Candidates keys: dict_keys(['content', 'finishReason', 'index'])
Content keys: dict_keys(['parts', 'role'])
Response time: 24.51 s


Processing 10.1002-14651858.CD010694.pub2 (2/380)...

Status code: 200
Response keys: dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])
Candidates keys: dict_keys(['content', 'finishReason', 'index', 'citationMetadata'])
Content keys: dict_keys(['parts', 'role'])
Response time: 19.78 s


Processing 10.1002-14651858.CD009416.pub2 (3/380)...

Status code: 200
Response keys: dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])
Candidates keys: dict_keys(['content', 'finishReason', 'index'])
Content keys: dict_keys(['parts', 'role'])
Response time: 17.55 s


Processing 10.1002-14651858.CD004104.pub4 (4/380)...

Status code: 200
Response keys: dict_keys(['candi

,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,Here is a Plain Language Summary of the study ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,Here is a Plain Language Summary of the biomed...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,Here is a Plain Language Summary of the study ...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,Here is a Plain Language Summary of the study ...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,Here is a Plain Language Summary of the biomed...


In [6]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_gemini_v2.csv"
df_check = pd.read_csv(ruta_csv)

df_check.info()
df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,Here is a Plain Language Summary of the study ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,Here is a Plain Language Summary of the biomed...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,Here is a Plain Language Summary of the study ...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,Here is a Plain Language Summary of the study ...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,Here is a Plain Language Summary of the biomed...


In [7]:
# Filtrar la fila faltante
faltante = df_check[df_check["gen_summary"].isnull()]
print(f"🔍 Faltan {len(faltante)} resúmenes.")
display(faltante[["name", "article"]])

🔍 Faltan 0 resúmenes.


,name,article


In [8]:
# Reprocesar solo esa fila
if not faltante.empty:
    for i, fila in faltante.iterrows():
        print(f"\nReprocesando {fila['name']}...\n")
        resumen, tiempo = call_gemini_api(prompt, fila['article'])
        df_check.loc[i, "gen_summary"] = resumen if resumen else ""
        if tiempo:
            print(f"Reparado en {tiempo:.2f} s")
    
    # Guardar nuevamente el CSV actualizado
    df_check.to_csv(ruta_csv, index=False, encoding="utf-8")
    print(f"Archivo actualizado: {os.path.abspath(ruta_csv)}")
else:
    print("No hay filas faltantes. Todo está completo.")

No hay filas faltantes. Todo está completo.
